# Глава 28. RAG и AI-агенты

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь объяснить, как работает RAG и чем он борется с галлюцинациями, описать, что такое AI-агент и чем он отличается от чат-бота, и собрать простейшего агента с одним инструментом.

В фазе 4 ты научился управлять LLM через промпты. Но современные ИИ-системы идут дальше: они подключают модель к внешним знаниям (RAG) и дают ей инструменты, которыми она пользуется сама (агенты). Это самые горячие темы 2024–2026 годов — и именно так строят реальные ИИ-продукты сегодня.

## RAG: LLM со своей базой знаний

**Аналогия: библиотекарь со справочником  💡**

Обычная LLM отвечает по памяти и может выдумывать (галлюцинации из главы 17). RAG (Retrieval-Augmented Generation) превращает её в библиотекаря, который перед ответом сходил к нужной полке и заглянул в справочник. Сначала система находит подходящий кусок из твоих документов, потом передаёт его модели вместе с вопросом. Ответ опирается на реальный текст, а не на «память».

Как это работает по шагам: (1) твои документы режутся на кусочки и превращаются в эмбеддинги — те самые «координаты смысла» из главы 17; (2) вопрос тоже превращается в эмбеддинг; (3) система находит кусочки, ближайшие по смыслу к вопросу (соседи на карте смыслов); (4) эти кусочки вставляются в промпт, и модель отвечает по ним. Именно так работают ИИ-помощники по документам компаний.

In [ ]:
# === Подготовка: функция ask() для запросов к LLM (как в главе 18) ===
# Запускать в Google Colab
import anthropic
from google.colab import userdata
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
def ask(prompt):
    msg = client.messages.create(model="claude-3-5-haiku-latest", max_tokens=200,
                                 messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

# Идея RAG на минимальном примере (без внешних библиотек)
# В реальности поиск кусочков делается по эмбеддингам автоматически

documents = {
    "оплата": "Оплата курса возможна помесячно или сразу за семестр.",
    "расписание": "Занятия проходят по субботам, два урока с перерывом.",
    "проект": "Финальный проект защищается на Demo Day перед родителями.",
}

def rag_answer(question):
    # 1. Находим самый релевантный кусок (упрощённо — по ключевому слову)
    chunk = None
    for topic, text in documents.items():
        if topic in question.lower():
            chunk = text
            break
    if chunk is None:
        return "В документах нет ответа."
    # 2. Отдаём найденный текст модели вместе с вопросом
    return ask(f"Ответь ТОЛЬКО по тексту: '{chunk}'. Вопрос: {question}")

print(rag_answer("Когда расписание занятий?"))

**Полезно знать  ℹ️**

В настоящих RAG-системах поиск кусочков делают не по ключевым словам, а по эмбеддингам через векторные базы данных (Chroma, FAISS) и фреймворки LangChain или LlamaIndex. Но идея ровно та же, что в примере: найти релевантный текст и дать его модели. Бесплатные эмбеддинги есть у Hugging Face — целую систему можно собрать без единого платного сервиса.

## AI-агенты: LLM, которая действует

**Аналогия: помощник с руками, а не только с языком  💡**

Чат-бот только разговаривает — отвечает текстом. AI-агент — как помощник, у которого есть не только язык, но и руки: он может воспользоваться инструментами. Спросил «сколько будет 847 × 293?» — агент не станет угадывать (LLM плохо считает!), а возьмёт калькулятор. Спросил про погоду — сходит в интернет. Агент сам решает, какой инструмент нужен, применяет его и возвращает результат.

Ключевое отличие: чат-бот генерирует текст, агент — совершает действия. Цикл работы агента: понять задачу → выбрать инструмент → применить → посмотреть результат → при необходимости повторить → дать финальный ответ. Именно агенты стоят за системами, которые бронируют билеты, пишут и запускают код, ищут информацию в интернете.

## Простейший агент с одним инструментом

Соберём мини-агента, который понимает, когда нужен калькулятор, и пользуется им. Это ядро идеи агентов без сложных фреймворков.

In [ ]:
# Мини-агент: решает, нужен ли калькулятор
def calculator(expression):
    return eval(expression)   # в учебных целях; в проде так не делают!

def agent(question):
    # Спрашиваем модель: нужен ли инструмент?
    answer = ask(
        f"Вопрос: {question}\n"
        "Если нужен точный расчёт — ответь ТОЛЬКО выражением "
        "вида 847*293. Иначе ответь сам."
    )
    # Если модель вернула математическое выражение — считаем инструментом
    if any(op in answer for op in ["*", "+", "-", "/"]):
        return f"Результат: {calculator(answer)}"
    return answer

print(agent("Сколько будет 847 умножить на 293?"))

Агент делегировал вычисление калькулятору, а не доверил его LLM, — и получил точный ответ. В настоящих агентах инструментов много (поиск, база данных, код), а решает, какой применить, сама модель. Фреймворки вроде LangChain автоматизируют этот цикл, но суть — та, что ты видишь здесь.

## Локальные модели: ИИ без интернета

Для приватных данных (медкарты, банковская тайна) и школ без стабильного интернета есть важная альтернатива — запускать модель локально, на своём компьютере. Инструменты Ollama и LM Studio позволяют скачать открытую модель (например, Llama или Mistral) и работать с ней офлайн, без отправки данных в чужие сервисы. Это медленнее облачных гигантов, но данные не покидают твой компьютер — критично для чувствительной информации и требований приватности, о которых говорят и в Таджикистане, и во всём мире.

## Задания

Базовый уровень

Собери мини-RAG по тексту любимого учебника или статьи: положи 3–4 абзаца в словарь и задай вопросы, ответы на которые есть только в тексте. Проверь: отвечает ли модель по тексту, а не выдумывает?

Со звёздочкой  ⭐

Расширь мини-агента вторым инструментом — например, «поиск факта» (возвращает заранее заданные факты). Научи агента выбирать между калькулятором и поиском в зависимости от вопроса. Опиши, чем твой агент отличается от обычного чат-бота.

Вопросы для обсуждения  💬

Чем AI-агент принципиально отличается от чат-бота?

Почему для точных вычислений агент использует калькулятор, а не саму LLM?

В каких случаях локальная модель (Ollama) лучше облачной, несмотря на меньшую мощность?

**Главная мысль части 7  🎯**

Ты вышел за пределы базового курса: умеешь проверять модели на справедливость, создавать сильные признаки, работать с несбалансированными данными и строить современные LLM-системы — RAG и агентов. Это темы, которые отличают специалиста, идущего в ногу со временем. Продолжай следить за областью — она развивается стремительно, и ты теперь понимаешь её язык.

**Проверь себя  ℹ️**

1. Как RAG уменьшает галлюцинации LLM?

2. Чем AI-агент отличается от чат-бота?

3. Когда стоит запускать модель локально (Ollama), а не в облаке?